In [1]:
import pandas as pd

In [6]:
import os

os.getcwd()

'/Users/npalaparthy/Desktop/fraud-rule-tuning-risk-scoring/notebooks'

In [7]:
os.listdir()

['01_data_review.ipynb']

In [8]:
pd.read_csv("../data/paysim_transactions.csv")

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.64,C1231006815,170136.00,160296.36,M1979787155,0.00,0.00,0,0
1,1,PAYMENT,1864.28,C1666544295,21249.00,19384.72,M2044282225,0.00,0.00,0,0
2,1,TRANSFER,181.00,C1305486145,181.00,0.00,C553264065,0.00,0.00,1,0
3,1,CASH_OUT,181.00,C840083671,181.00,0.00,C38997010,21182.00,0.00,1,0
4,1,PAYMENT,11668.14,C2048537720,41554.00,29885.86,M1230701703,0.00,0.00,0,0
...,...,...,...,...,...,...,...,...,...,...,...
6362615,743,CASH_OUT,339682.13,C786484425,339682.13,0.00,C776919290,0.00,339682.13,1,0
6362616,743,TRANSFER,6311409.28,C1529008245,6311409.28,0.00,C1881841831,0.00,0.00,1,0
6362617,743,CASH_OUT,6311409.28,C1162922333,6311409.28,0.00,C1365125890,68488.84,6379898.11,1,0
6362618,743,TRANSFER,850002.52,C1685995037,850002.52,0.00,C2080388513,0.00,0.00,1,0


In [11]:
df = pd.read_csv("../data/paysim_transactions.csv")

df.shape

(6362620, 11)

In [12]:
df.columns

Index(['step', 'type', 'amount', 'nameOrig', 'oldbalanceOrg', 'newbalanceOrig',
       'nameDest', 'oldbalanceDest', 'newbalanceDest', 'isFraud',
       'isFlaggedFraud'],
      dtype='object')

In [13]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6362620 entries, 0 to 6362619
Data columns (total 11 columns):
 #   Column          Dtype  
---  ------          -----  
 0   step            int64  
 1   type            object 
 2   amount          float64
 3   nameOrig        object 
 4   oldbalanceOrg   float64
 5   newbalanceOrig  float64
 6   nameDest        object 
 7   oldbalanceDest  float64
 8   newbalanceDest  float64
 9   isFraud         int64  
 10  isFlaggedFraud  int64  
dtypes: float64(5), int64(3), object(3)
memory usage: 534.0+ MB


In [14]:
df["isFraud"].value_counts()

isFraud
0    6354407
1       8213
Name: count, dtype: int64

In [15]:
df["isFraud"].value_counts(normalize=True)

isFraud
0    0.998709
1    0.001291
Name: proportion, dtype: float64

In [16]:
df["type"].value_counts()

type
CASH_OUT    2237500
PAYMENT     2151495
CASH_IN     1399284
TRANSFER     532909
DEBIT         41432
Name: count, dtype: int64

In [17]:
transaction_type_summary = df.groupby("type").agg(
    total_transactions=("isFraud", "count"),
    fraud_transactions=("isFraud", "sum")
).reset_index()

transaction_type_summary["fraud_rate"] = (
    transaction_type_summary["fraud_transactions"] 
    / transaction_type_summary["total_transactions"]
)

transaction_type_summary = transaction_type_summary.sort_values(
    "fraud_rate", 
    ascending=False
)

transaction_type_summary

,type,total_transactions,fraud_transactions,fraud_rate
4,TRANSFER,532909,4097,0.007688
1,CASH_OUT,2237500,4116,0.001840
0,CASH_IN,1399284,0,0.000000
2,DEBIT,41432,0,0.000000
3,PAYMENT,2151495,0,0.000000


In [18]:
transaction_type_summary.to_csv(
    "../outputs/transaction_type_fraud_summary.csv", 
    index=False
)